In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

import utils
import billing_calcs

from const import *
from datetime import datetime, time

In [ ]:
half_hourly_usage_df = utils.get_half_hourly_usage_data()
monthly_usage_df = utils.get_monthly_usage_data()

display(half_hourly_usage_df.head(), f"{len(half_hourly_usage_df)} rows")
display(monthly_usage_df.head(), f"{len(monthly_usage_df)} rows")

In [ ]:
# condense overlapping DST hours
duplicated_idx = half_hourly_usage_df[START_DT_COL].duplicated(keep='first')
adjustment_df = half_hourly_usage_df.loc[duplicated_idx].groupby(START_DT_COL)[UNITS_COL].sum()
half_hourly_usage_df = half_hourly_usage_df.loc[~duplicated_idx]
update_idx = half_hourly_usage_df[START_DT_COL].isin(adjustment_df.index)
half_hourly_usage_df.loc[update_idx, UNITS_COL] = half_hourly_usage_df.loc[update_idx, UNITS_COL] + adjustment_df

In [ ]:
half_hourly_usage_df['time'] = half_hourly_usage_df[START_DT_COL].dt.time
half_hourly_usage_df['day'] = half_hourly_usage_df[START_DT_COL].dt.day_name()
half_hourly_usage_df['date'] = half_hourly_usage_df[START_DT_COL].dt.date

In [ ]:
mercury_variable_rate = 0.2749
mercury_fixed_rate = 1.5

supa_variable_rate = 0.2351
supa_fixed_rate = 1.725

ek_off_peak_rate = 0.3357
ek_peak_rate = 0.3731
ek_fixed_rate = 1.72

ek_off_peak_1_start = time(9,0,0)
ek_off_peak_1_end = time(17,00)
ek_off_peak_2_start = time(21,00)
ek_off_peak_2_end = time(7,00)

In [ ]:
mercury_calcs = []
ek_calcs = []
supa_calcs = []
contact_lu_calcs = []
contact_gc_calcs = []

best_hour_of_power = []

for index, monthly_use in monthly_usage_df.iterrows():
    
    print(
        "Billing Period:",
        monthly_use['start_dt'], 
        monthly_use['end_dt']
    )

    #filter to billing periods and reshape
    billing_units_df = (
        half_hourly_usage_df.loc[
            (
                half_hourly_usage_df['date'] 
                > monthly_use['start_dt'].date()
            )  
            & (
                half_hourly_usage_df['date'] 
                <= monthly_use['end_dt'].date()
            )
        ].pivot(index=['date','day'], columns='time', values=UNITS_COL)
    )

    # calc mercury bill
    mercury_calc = billing_calcs.calc_simple_bill(
        billing_units_df.copy(), 
        fixed_rate=mercury_fixed_rate, 
        variable_rate=mercury_variable_rate
    )
    mercury_calcs.append(mercury_calc)

    # calc supa bill
    supa_calc = billing_calcs.calc_simple_bill(
        billing_units_df.copy(), 
        fixed_rate=supa_fixed_rate, 
        variable_rate=supa_variable_rate
    )
    supa_calcs.append(supa_calc)

    # calc contact low user
    contact_lu_calc = billing_calcs.calc_contact_low_user_bill(
        billing_units_df.copy()
    )
    contact_lu_calcs.append(contact_lu_calc)

    # calc contact good charge
    contact_gc_calc = billing_calcs.calc_contact_good_charge_bill(
        billing_units_df.copy()
    )
    contact_gc_calcs.append(contact_gc_calc)

    # calc electric kiwi bill with all hour of power options 
    # and then return minimum
    ek_calc_i = []
    times = []
    
    for hour in range(24):
        for mins in [0,30]:
        
            ek_calc_i_hour = billing_calcs.calc_electric_kiwi_bill(
                billing_units_df.copy(),
                off_peak_rate=ek_off_peak_rate,
                peak_rate=ek_peak_rate,
                fixed_rate=ek_fixed_rate,
                off_peak_1_start=ek_off_peak_1_start,
                off_peak_1_end=ek_off_peak_1_end,
                off_peak_2_start=ek_off_peak_2_start,
                off_peak_2_end=ek_off_peak_2_end,
                hour_of_power_start=time(hour, mins)
            )
    
            ek_calc_i.append(ek_calc_i_hour)
            times.append(time(hour, mins)) 

    ek_calcs.append(np.min(ek_calc_i))
    best_hour_of_power.append(
        times[np.argmin(ek_calc_i)]
    )

comparison = pd.DataFrame(
    {
        "Mercury - Low User": mercury_calcs,
        "Electric Kiwi - Go 250 (Low User)": ek_calcs,
        "Supa - Purple Pancake": supa_calcs,
        "Contact - Low User": contact_lu_calcs,
        "Contact - Good Charge": contact_gc_calcs,
    }
)

comparison["Billing Period Start"] = monthly_usage_df['start_dt']

In [ ]:
fig = px.line(
    comparison,
    x="Billing Period Start",
    y=[
        "Mercury - Low User", 
        "Electric Kiwi - Go 250 (Low User)", 
        "Supa - Purple Pancake",
        "Contact - Low User",
        "Contact - Good Charge",
    ]
)
fig.update_yaxes(title_text="Retrospective Cost $")
fig.update_layout(legend_title_text='Power Plan')

fig.show()

display(comparison.set_index("Billing Period Start").sum(axis=0))
print(best_hour_of_power)